[![-Badge](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/dpo_orpo_alignment_tutorial.ipynb)

# Qwen3 / Qwen2.5 Preference Optimization (DPO & ORPO) Alignment Tutorial

This notebook provides a comprehensive end-to-end guide to aligning language models using **Direct Preference Optimization (DPO)** and **Odds Ratio Preference Optimization (ORPO)** in MaxText. By leveraging the unified `DPOTrainer` integration, you can easily perform preference learning on Cloud TPUs and GPUs.

## Overview of DPO and ORPO

Preference optimization is the final phase of training state-of-the-art alignment models. Instead of training a complex Reinforcement Learning pipeline (which requires an actor, critic, and reference model), modern preference learning techniques optimize models directly on pairs of preferred and dispreferred responses.

### 1. Direct Preference Optimization (DPO)
DPO optimizes a binary classification loss over preference pairs. It computes the log-probabilities of the policy model for both chosen and rejected responses, and scales the optimization using a reference model:
$$\mathcal{L}_{\text{DPO}}(\theta; \theta_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\theta_{\text{ref}}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\theta_{\text{ref}}}(y_l|x)} \right) \right]$$
Where $y_w$ is the chosen response, $y_l$ is the rejected response, $\pi_\theta$ is the policy, and $\pi_{\theta_{\text{ref}}}$ is the frozen reference model.

### 2. Odds Ratio Preference Optimization (ORPO)
ORPO integrates SFT and preference alignment into a single joint objective. It optimizes the standard language modeling loss (SFT loss) on the chosen response, and uses an odds ratio penalty to discourage the model from generating the rejected response:
$$\mathcal{L}_{\text{ORPO}}(\theta) = \mathcal{L}_{\text{SFT}}(\theta) - \lambda \cdot \log \text{OddsRatio}_\theta(y_w, y_l | x)$$
**Why ORPO?** Because it does not require a reference model, ORPO saves **50% memory** compared to DPO, allowing you to train larger models on the same hardware.

---

### What We Will Accomplish in this Notebook
1.  **Environment Setup:** Detect environment (Colab vs. TPU VM) and install dependencies.
2.  **Interactive Parameters:** Configure parameters in a single, easy-to-customize cell.
3.  **Dataset Setup:** Stream and inspect preference data.
4.  **On-the-Fly Checkpoint Prep:** Download and shard Qwen3/Qwen2.5 models.
5.  **Alignment Training:** Initialize the `DPOTrainer` and run preference alignment.
6.  **Inference Checkpoint Export:** Export weights to standard Flax/Linen format for high-throughput serving.

In [ ]:
try:
  import google.colab
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
  print("Running the notebook on a local TPU VM or GPU Jupyter instance")
  IN_COLAB = False

## 1. Installation & Setup

If running in Google Colab, we will clone MaxText and install the post-training alignment suite along with TPU/GPU acceleration prerequisites. 

**Note on Colab session restarts:** When installing packages on Google Colab, JAX and other deep learning runtimes require a session restart. Run the cell below, wait for it to complete, and restart your runtime (`Runtime > Restart session` or `Ctrl + M .`) before executing the rest of the notebook.

In [ ]:
if IN_COLAB:
    # Clone MaxText
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv for fast compilation/resolution
    !pip install uv
    
    # Install MaxText with post-training dependencies
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps
    
    print("\n✓ Installation complete. Please restart your Colab session now.")

## 2. Imports & Device Initialization

Let's import JAX, MaxText configuration layers, and verification helpers, and initialize the hardware devices.

In [ ]:
import datetime
import os
import sys
from etils import epath
import jax
import optax
from datasets import load_dataset

from maxtext.configs import pyconfig
from maxtext.utils import max_logging, maxtext_utils, model_creation_utils
from maxtext.trainers.post_train.dpo import train_dpo, hooks
from maxtext.utils.globals import MAXTEXT_PKG_DIR

# Suppress noisy TF/jax logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print(f"JAX version: {jax.__version__}")
print(f"Available JAX backends: {jax.devices()}")
print(f"Process index: {jax.process_index()} / {jax.process_count()}")

### Authenticating with Hugging Face

Hugging Face access tokens are required to download instruction-tuned weights (like Qwen or Gemma). The cell below launches an interactive login prompt.

In [ ]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    from huggingface_hub import login
    # Attempts login via CLI if token is present, else prompts interactively
    login()

## 3. Customization Parameters

All parameters are consolidated in the cell below. You can toggle between DPO and ORPO, change the underlying model sizing, select training steps, and adjust hyperparameters.

> [!IMPORTANT]
> **Disk Space & Checkpoints:** Model checkpoints during training can be extremely large. To prevent running out of local disk space on a TPU VM, we highly recommend configuring `BASE_OUTPUT_DIRECTORY` to a **Google Cloud Storage (GCS)** path (e.g., `gs://your-bucket/align_output`).

In [ ]:
# @title Optimization Parameters { run: "auto" }

# Selection of Preference Optimization algorithm
ALGORITHM = "dpo" # @param ["dpo", "orpo"]

# Sizing and name of target model configuration
MODEL_NAME = "qwen3-0.6b" # @param ["qwen3-0.6b", "gemma2-2b", "llama3-8b"]
TOKENIZER_PATH = "Qwen/Qwen3-0.6B" # @param {type:"string"}

# Preference learning training split parameters
STEPS = 15 # @param {type:"integer"}
LEARNING_RATE = 1e-6 # @param {type:"number"}
BATCH_SIZE = 1 # @param {type:"integer"}
MAX_TARGET_LENGTH = 1024 # @param {type:"integer"}
EVAL_INTERVAL = 10 # @param {type:"integer"}

# Model hyperparameters
BETA = 0.1 # @param {type:"number"}
ORPO_LAMBDA = 0.1 # @param {type:"number"}
LABEL_SMOOTHING = 0.0 # @param {type:"number"}

# Outputs
# We highly recommend using a GCS path (gs://bucket-name/path) to save local disk space
BASE_OUTPUT_DIRECTORY = "/tmp/align_output" # @param {type:"string"}
RUN_NAME = f"{ALGORITHM}-{MODEL_NAME}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"✓ Parameter configuration initialized.")
print(f"  Output Path: {BASE_OUTPUT_DIRECTORY}/{RUN_NAME}")

### Instantiating MaxText Configuration

We load the baseline MaxText DPO configuration, merging our customization parameters programmatically.

In [ ]:
# Construct configuration argument vector
config_argv = [
    "",
    f"{MAXTEXT_PKG_DIR}/configs/post_train/dpo.yml",
    f"run_name={RUN_NAME}",
    f"base_output_directory={BASE_OUTPUT_DIRECTORY}",
    f"model_name={MODEL_NAME}",
    f"tokenizer_path={TOKENIZER_PATH}",
    "dataset_type=hf",
    "hf_path=argilla/distilabel-intel-orca-dpo-pairs",
    "hf_eval_split=train",
    "train_split=train",
    f"steps={STEPS}",
    f"learning_rate={LEARNING_RATE}",
    f"per_device_batch_size={BATCH_SIZE}",
    f"max_target_length={MAX_TARGET_LENGTH}",
    f"eval_interval={EVAL_INTERVAL}",
    "enable_checkpointing=True",
    "checkpoint_period=15",
    "log_config=False",
    "use_dpo=True",
    f"dpo.algo={ALGORITHM}",
    f"dpo.dpo_beta={BETA}",
    f"dpo.orpo_lambda={ORPO_LAMBDA}",
    f"dpo.dpo_label_smoothing={LABEL_SMOOTHING}",
    "enable_goodput_recording=False",
    "monitor_goodput=False",
]

config = pyconfig.initialize(config_argv)
print(f"MaxText Configuration parsed.")
print(f"  Target Steps: {config.steps}")
print(f"  DPO Algorithm: {config.dpo.algo.upper()}")
print(f"  Beta parameter: {config.dpo.dpo_beta}")

## 4. Dataset Preview

Let's stream and print two samples from our preference dataset `argilla/distilabel-intel-orca-dpo-pairs`. DPO and ORPO models expect preference pairs containing a common prompt, a high-quality chosen response, and a lower-quality rejected response.

In [ ]:
print("Streaming sample pairs...")
preview_ds = load_dataset(config.hf_path, split="train", streaming=True)
samples = list(preview_ds.take(2))

for idx, sample in enumerate(samples):
    print(f"\n=== Preference Pair {idx+1} ===")
    print(f"PROMPT (first 300 chars):\n{sample['input'][:300]}...")
    print(f"\nCHOSEN RESPONSE (first 300 chars):\n{sample['chosen'][:300]}...")
    print(f"\nREJECTED RESPONSE (first 300 chars):\n{sample['rejected'][:300]}...")
    print("=================================")

## 5. Run Preference Optimization Training

We will execute the preference optimization training loop by calling `train_dpo.train`. This automatically handles loading/sharding the pre-trained checkpoint, configuring the reference model (for DPO), initializing optimization schedules, and running preference alignment.

During training, watch the logs for the following key indicators:
*   `dpo_loss`/`sft_loss`: Should steadily decline.
*   `rewards/accuracy`: Measures the proportion of preference pairs where the model successfully assigns higher probability to the preferred chosen response than the dispreferred rejected response. In qualification runs, this rises to **>75%** accuracy.
*   `rewards/margin`: The difference in logratios. It should become positive.

In [ ]:
print(f"Starting {config.dpo.algo.upper()} training...")
trainer, mesh = train_dpo.train(config)
print(f"✓ {config.dpo.algo.upper()} training successfully completed.")

## 6. Exporting Inference Checkpoints

The checkpoints saved during training are stateful Flax NNX parameters containing training namespaces (like the prefix `base.`) and optimization tracking branches.
To make these compatible with high-performance serving engines (e.g., JetStream), we strip the optimization metadata and export standard, replicated Linen-compatible parameters.

In [ ]:
import shutil
from flax import nnx

# Retrieve active saved trainer checkpoints
checkpoint_dir = config.checkpoint_dir
print(f"Analyzing saved training state under: {checkpoint_dir}")

# Prepare destination for clean, flat inference parameters
inference_dir = epath.Path(config.base_output_directory) / config.run_name / "inference_ckpt"
inference_dir.mkdir(parents=True, exist_ok=True)

# Convert and replicated sharding export
state_params = nnx.state(trainer.model.base, nnx.Param)
replicated = jax.sharding.NamedSharding(mesh, jax.sharding.PartitionSpec())

def nnx_state_to_replicated_linen_dict(state_node):
  if isinstance(state_node, nnx.Variable):
    # Convert leaf weights to replicated CPU arrays
    return jax.device_put(state_node.value, replicated)
  if isinstance(state_node, (nnx.State, dict)):
    return {k: nnx_state_to_replicated_linen_dict(v) for k, v in state_node.items()}
  return state_node

print("Converting state structure...")
clean_params = nnx_state_to_replicated_linen_dict(state_params)

import orbax.checkpoint as ocp
ckptr = ocp.Checkpointer(ocp.PyTreeCheckpointHandler())
ckptr.save(
    inference_dir / "0" / "items",
    clean_params
)
print(f"✓ Replicated Linen-compatible inference parameters saved to: {inference_dir}/0/items")

## Conclusion

You have aligned a language model using Direct Preference Optimization (DPO) or Odds Ratio Preference Optimization (ORPO) on Google TPUs / GPUs. The resulting inference checkpoint is ready to be loaded directly into high-performance serving infrastructures like JetStream or standard Hugging Face pipelines!